In [1]:
import os
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import *

# Adding AWS S3 Minio configs
sparkConf = (
    SparkConf()
    .set("spark.jars.ivy","/home/brijeshdhaker/.ivy2")
    .set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .set("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.797,io.delta:delta-spark_2.12:3.3.2")
    .set("spark.executor.heartbeatInterval", "300000")
    .set("spark.network.timeout", "400000")
    .set("spark.hadoop.fs.defaultFS", "s3a://defaultfs/")
    .set("spark.hadoop.fs.s3a.endpoint", "http://minio.sandbox.net:9010")
    .set("spark.hadoop.fs.s3a.access.key", "pgm2H2bR7a5kMc5XCYdO")
    .set("spark.hadoop.fs.s3a.secret.key", "zjd8T0hXFGtfemVQ6AH3yBAPASJNXNbVSx5iddqG")
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.delta.enableFastS3AListFrom", "true")
    #.set("spark.eventLog.enabled", "true")
    #.set("spark.eventLog.dir", "file:///apps/var/logs/spark-events")
)

# configure the SparkSession with the configure_spark_with_delta_pip() utility function in Delta Lake:
builder = SparkSession.builder.appName("spark-deltalake-timetravel").master("local[*]").config(conf=sparkConf)
spark = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.hadoop:hadoop-aws:3.3.4"]).getOrCreate()

#
spark.sparkContext.setLogLevel('ERROR')
spark

#
# 
#
def display(df):
    df.show(truncate=False)

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/brijeshdhaker/.ivy2/cache
The jars for the packages stored in: /home/brijeshdhaker/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6d4390b2-cbac-4c17-b858-57b260998aeb;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in local-m2-cache
	found io.delta#delta-storage;3.3.2 in local-m2-cache
	found org.antlr#antlr4-runtime;4.9.3 in local-m2-cache
	found org.apache.hadoop#hadoop-aws;3.3.4 in local-m2-cache
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in local-m2-cache
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in local-m2-cache
:: resolution report :: resolve 166ms :: artifacts dl 13ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from local-m2-cache in [default]
	io.delta#delta-spark_2.12;3.3.2 from local-m2-cache in [default]
	io.delta#delta-storage;3.3.2 from local-m2-cache in [defaul

In [2]:
%load_ext sql

In [3]:
%sql spark

##### Delete Existing Delta Table

In [6]:
%%sql

DROP TABLE IF EXISTS delta.`/deltalake/invoices_ttv`;

Running query in 'SparkSession'

++
||
++
++

In [ ]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://defaultfs/deltalake/invoices_ttv --recursive

delete: s3://defaultfs/deltalake/invoices_ttv/_delta_log/00000000000000000000.crc
delete: s3://defaultfs/deltalake/invoices_ttv/_delta_log/00000000000000000000.json
delete: s3://defaultfs/deltalake/invoices_ttv/_delta_log/00000000000000000001.crc
delete: s3://defaultfs/deltalake/invoices_ttv/_delta_log/00000000000000000002.crc
delete: s3://defaultfs/deltalake/invoices_ttv/_delta_log/00000000000000000001.json
delete: s3://defaultfs/deltalake/invoices_ttv/_delta_log/00000000000000000002.json
delete: s3://defaultfs/deltalake/invoices_ttv/_delta_log/00000000000000000003.crc
delete: s3://defaultfs/deltalake/invoices_ttv/_delta_log/00000000000000000003.json
delete: s3://defaultfs/deltalake/invoices_ttv/_delta_log/00000000000000000004.json
delete: s3://defaultfs/deltalake/invoices_ttv/_delta_log/00000000000000000004.checkpoint.parquet
delete: s3://defaultfs/deltalake/invoices_ttv/_delta_log/00000000000000000005.checkpoint.parquet
delete: s3://defaultfs/deltalake/invoices_ttv/_delta_log/000000

##### Create Delta Table

In [ ]:
df = spark.read.format("parquet").load("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet")
df.printSchema()

#
display(df)

# Save as delta table into Minio S3
df.write.format('delta').save('/deltalake/invoices_ttv')

root
 |-- customer_id: integer (nullable = true)
 |-- invoice_no: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- invoice_date: date (nullable = true)
 |-- shopping_mall: string (nullable = true)
 |-- _rescued_data: string (nullable = true)

+-----------+----------+------+---+---------------+--------+-------+--------------+------------+----------------+-------------+
|customer_id|invoice_no|gender|age|category       |quantity|price  |payment_method|invoice_date|shopping_mall   |_rescued_data|
+-----------+----------+------+---+---------------+--------+-------+--------------+------------+----------------+-------------+
|1          |I178410   |Male  |61 |Clothing       |5       |1500.4 |Credit Card   |2021-11-26  |Metrocity       |NULL         |
|2          |I158163   |Ma

In [4]:
%%sql

CREATE OR REPLACE TABLE delta.`/deltalake/invoices_ttv` USING  DELTA
AS SELECT * FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet`;

Running query in 'SparkSession'

++
||
++
++

In [5]:
%%sql

SELECT * FROM delta.`/deltalake/invoices_ttv`
LIMIT 5;

Running query in 'SparkSession'

5 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11
1,I178410,Male,61,Clothing,5,1500.4,Credit Card,2021-11-26,Metrocity,None
2,I158163,Male,34,Shoes,2,1200.34,Cash,2023-03-03,Kanyon,None
3,I262373,Male,44,Toys,3,107.52,Credit Card,2022-12-01,Cevahir AVM,None
4,I334895,Male,25,Food & Beverage,5,26.15,Cash,2021-08-15,Kanyon,None
5,I202043,Female,21,Toys,1,35.84,Credit Card,2021-07-25,Metrocity,None


In [13]:
%%sql

DELETE FROM delta.`/deltalake/invoices_ttv`
WHERE customer_id = 1;

Running query in 'SparkSession'

1 rows affected.

Field 1
1


In [14]:
%%sql

UPDATE delta.`/deltalake/invoices_ttv`
SET quantity = 25
WHERE customer_id = 5; 

Running query in 'SparkSession'

1 rows affected.

Field 1
1


In [15]:
%%sql

INSERT INTO delta.`/deltalake/invoices_ttv`
SELECT * FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet`

Running query in 'SparkSession'

++
||
++
++

In [16]:
%%sql


SELECT COUNT(*) FROM delta.`/deltalake/invoices_ttv`
WHERE customer_id > 100; 

Running query in 'SparkSession'

1 rows affected.

Field 1
100


In [17]:
%%sql

DESCRIBE HISTORY delta.`/deltalake/invoices_ttv`;

Running query in 'SparkSession'

4 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
3,2026-03-31 13:15:01,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,2,Serializable,True,"{'numOutputRows': '100', 'numOutputBytes': '5745', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
2,2026-03-31 13:14:11,None,None,UPDATE,"{'predicate': '[""(customer_id#2784 = 5)""]'}",None,None,None,1,Serializable,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFiles': '1', 'executionTimeMs': '1056', 'numDeletionVectorsRemoved': '0', 'numUpdatedRows': '1', 'numRemovedFiles': '1', 'rewriteTimeMs': '162', 'numRemovedBytes': '5749', 'scanTimeMs': '893', 'numCopiedRows': '98', 'numDeletionVectorsAdded': '0', 'numAddedChangeFiles': '0', 'numAddedBytes': '5754'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
1,2026-03-31 13:14:01,None,None,DELETE,"{'predicate': '[""(customer_id#2070 = 1)""]'}",None,None,None,0,Serializable,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFiles': '1', 'executionTimeMs': '634', 'numDeletionVectorsRemoved': '0', 'numRemovedFiles': '1', 'rewriteTimeMs': '176', 'numRemovedBytes': '5768', 'scanTimeMs': '457', 'numCopiedRows': '99', 'numDeletionVectorsAdded': '0', 'numAddedChangeFiles': '0', 'numDeletedRows': '1', 'numAddedBytes': '5749'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
0,2026-03-31 13:08:37,None,None,WRITE,"{'mode': 'ErrorIfExists', 'partitionBy': '[]'}",None,None,None,None,Serializable,True,"{'numOutputRows': '100', 'numOutputBytes': '5768', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [18]:
%%sql

SELECT * FROM delta.`/deltalake/invoices_ttv` VERSION AS OF 0
WHERE customer_id = 1;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11
1,I178410,Male,61,Clothing,5,1500.4,Credit Card,2021-11-26,Metrocity,None


In [46]:
%%sql

SELECT * FROM delta.`/deltalake/invoices_ttv` TIMESTAMP AS OF '2026-03-31T07:45:09.000+00:00'
WHERE customer_id = 1;

Running query in 'SparkSession'

++
||
++
++

In [47]:
%%sql

RESTORE TABLE delta.`/deltalake/invoices_ttv` TO VERSION AS OF 0; 
-- RESTORE TABLE delta.`/deltalake/invoices_ttv` TO TIMESTAMP AS OF '2025-02-02T07:08:09.000+00:00';

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6
5768,1,0,0,0,0


In [26]:
%%sql

SELECT * FROM delta.`/deltalake/invoices_ttv`
WHERE customer_id = 1;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11
1,I178410,Male,61,Clothing,5,1500.4,Credit Card,2021-11-26,Metrocity,None


In [27]:
%%sql

DESCRIBE HISTORY delta.`/deltalake/invoices_ttv`;

Running query in 'SparkSession'

5 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
4,2026-03-31 13:19:57,None,None,RESTORE,"{'version': '0', 'timestamp': None}",None,None,None,3,Serializable,False,"{'removedFilesSize': '11499', 'tableSizeAfterRestore': '5768', 'numRemovedFiles': '2', 'restoredFilesSize': '5768', 'numOfFilesAfterRestore': '1', 'numRestoredFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
3,2026-03-31 13:15:01,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,2,Serializable,True,"{'numOutputRows': '100', 'numOutputBytes': '5745', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
2,2026-03-31 13:14:11,None,None,UPDATE,"{'predicate': '[""(customer_id#2784 = 5)""]'}",None,None,None,1,Serializable,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFiles': '1', 'executionTimeMs': '1056', 'numDeletionVectorsRemoved': '0', 'numUpdatedRows': '1', 'numRemovedFiles': '1', 'rewriteTimeMs': '162', 'numRemovedBytes': '5749', 'scanTimeMs': '893', 'numCopiedRows': '98', 'numDeletionVectorsAdded': '0', 'numAddedChangeFiles': '0', 'numAddedBytes': '5754'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
1,2026-03-31 13:14:01,None,None,DELETE,"{'predicate': '[""(customer_id#2070 = 1)""]'}",None,None,None,0,Serializable,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFiles': '1', 'executionTimeMs': '634', 'numDeletionVectorsRemoved': '0', 'numRemovedFiles': '1', 'rewriteTimeMs': '176', 'numRemovedBytes': '5768', 'scanTimeMs': '457', 'numCopiedRows': '99', 'numDeletionVectorsAdded': '0', 'numAddedChangeFiles': '0', 'numDeletedRows': '1', 'numAddedBytes': '5749'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
0,2026-03-31 13:08:37,None,None,WRITE,"{'mode': 'ErrorIfExists', 'partitionBy': '[]'}",None,None,None,None,Serializable,True,"{'numOutputRows': '100', 'numOutputBytes': '5768', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [0]:
-- 3	2025-02-02T07:09:57.000+00:00
--   2025-02-02T07:09:35.000+00:00
-- 2 2025-02-02T07:09:31.000+00:00

In [48]:
%%sql

SELECT * FROM delta.`/deltalake/invoices_ttv` TIMESTAMP AS OF '2026-03-31 13:19:57' -- version 2
WHERE customer_id = 5;  

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11
5,I202043,Female,21,Toys,25,35.84,Credit Card,2021-07-25,Metrocity,None


In [49]:
%%sql

SELECT * FROM delta.`/deltalake/invoices_ttv` -- version 0
WHERE customer_id = 5;  

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11
5,I202043,Female,21,Toys,1,35.84,Credit Card,2021-07-25,Metrocity,None


In [50]:

from pyspark.sql.functions import col
df = spark.read.option("versionAsOf", "1").table("delta.`/deltalake/invoices_ttv`")
display(df.filter(col("customer_id") == 1))

+-----------+----------+------+---+--------+--------+-----+--------------+------------+-------------+-------------+
|customer_id|invoice_no|gender|age|category|quantity|price|payment_method|invoice_date|shopping_mall|_rescued_data|
+-----------+----------+------+---+--------+--------+-----+--------------+------------+-------------+-------------+
+-----------+----------+------+---+--------+--------+-----+--------------+------------+-------------+-------------+



In [31]:

df = spark.read.option("versionAsOf", "2").table("delta.`/deltalake/invoices_ttv`")
display(df.filter(col("customer_id") == 5))

+-----------+----------+------+---+--------+--------+-----+--------------+------------+-------------+-------------+
|customer_id|invoice_no|gender|age|category|quantity|price|payment_method|invoice_date|shopping_mall|_rescued_data|
+-----------+----------+------+---+--------+--------+-----+--------------+------------+-------------+-------------+
|5          |I202043   |Female|21 |Toys    |25      |35.84|Credit Card   |2021-07-25  |Metrocity    |NULL         |
+-----------+----------+------+---+--------+--------+-----+--------------+------------+-------------+-------------+



In [51]:

df = spark.read.option("timestampAsOf", "2026-03-31 13:19:57").table("delta.`/deltalake/invoices_ttv`")
display(df.filter(col("customer_id") == 5))

+-----------+----------+------+---+--------+--------+-----+--------------+------------+-------------+-------------+
|customer_id|invoice_no|gender|age|category|quantity|price|payment_method|invoice_date|shopping_mall|_rescued_data|
+-----------+----------+------+---+--------+--------+-----+--------------+------------+-------------+-------------+
|5          |I202043   |Female|21 |Toys    |25      |35.84|Credit Card   |2021-07-25  |Metrocity    |NULL         |
+-----------+----------+------+---+--------+--------+-----+--------------+------------+-------------+-------------+

